# Data Health Audit & Preparation

## Important Note

This notebook `/notebooks/3_data_health_audit_&_prep_global.ipynb` is different than the sub-sampled version `/notebooks/3_data_health_audit_&_prep_only.ipynb` in terms of structure and purpose. 

Purpose: The sub-sampled version aims to check the data health and explore potential issue in various aspect of the dataset. This one aims to verify if the issues/trend/patterns found in the sub-sampled counterpart will be similar to this version. The similar pattern exhibits from two notebooks will indirectly indicates that the subsampling strategy is unbiased and effective, and vice versa.

Structure: The subsampled version will explicitly write down the logic as you audit the data health, whereas this version will put most workflows in `/src/utils/data_health_audit_helper.py`, and the audit logic will call a specific function to perform and verify health check. 

In [1]:
# Make sure you are at the parent directory
from pathlib import Path
import sys

# Define MODE
MODE = "EXPANSE" # "EXPANSE" is the only mode
if MODE.upper() != "EXPANSE":
    raise Exception("Invalid mode, the only acceptible is 'EXPANSE'")

# Root path by MODE
PROJECT_ROOT_BY_MODE = {
    "COLAB": Path("/content/drive/MyDrive/DSC 288R/Project"),
    "EXPANSE": Path("/home/bguo3/bguo3/DSC-288R-Capstone-Final-Project"),
    "LOCAL": Path("/Users/steveg/Desktop/DSC-288R-Capstone-Final-Project"),
}
ROOT = PROJECT_ROOT_BY_MODE[MODE.upper()]

# Add the root path to global system
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Print the root path
print("MODE:", MODE)
print("PROJECT_ROOT:", ROOT)

MODE: EXPANSE
PROJECT_ROOT: /home/bguo3/bguo3/DSC-288R-Capstone-Final-Project


## Import Modules

In [2]:
import os

from src.utils.pyspark_utils import create_spark_session, memory_count
from src.utils.paths_utils import ProjectPaths
from src.utils.io_utils import read_spark_parquet, write_spark_parquet
from src.pipelines.data_health_audit import print_section, format_report
import src.pipelines.data_health_audit as audit
import src.pipelines.data_prep as prep

# Data health audit functions under audit includes:
    # structure_report
    # schema_audit
    # null_report
    # consistency_report
    # validity_report
    # anomaly_report
    # noise_report
    # uniqueness_report
    # duplicate_report
    
    # print_section
    # format_report

Matplotlib created a temporary cache directory at /scratch/bguo3/job_49085282/matplotlib-6joo4xm3 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Read Full Data Parquet File

In [3]:
# Set up for Spark app & resource allocation
spark = create_spark_session("steam_reviews_data_health_check")

# Load data
paths = ProjectPaths(MODE)
raw_df = read_spark_parquet(spark, paths.full_parquet)

Read Spark parquet from: /expanse/lustre/scratch/bguo3/temp_project/steam_reviews/full_parquet


## Health Audit - Dataset Structure

In [4]:
ROW_COUNT = raw_df.count()

## Shape
# 1. (# of rows, # of columns, # of cells)
print_section("1. (# of rows, # of columns, # of cells)")
display(audit.structure_report(raw_df, ROW_COUNT))

## 2. Concise summary on columns details
print_section("2. Concise summary on columns details")
display(audit.schema_audit(raw_df))
raw_df.printSchema()

# 3. Memory count
print_section("3. Memory count")
memory_count(raw_df)


1. (# of rows, # of columns, # of cells)


,row_count,column_count,cell_count
0,113885601,18,2049940818



2. Concise summary on columns details


,column_name,present_in_df,expected_type,actual_type,matches_expected_family
0,author_steamid,True,bigint,bigint,True
1,appid,True,int,int,True
2,author_num_games_owned,True,int,int,True
3,author_num_reviews,True,int,int,True
4,author_playtime_forever,True,int,int,True
5,author_playtime_last_two_weeks,True,int,int,True
6,author_playtime_at_review,True,int,int,True
7,author_last_played,True,bigint,bigint,True
8,review,True,string,string,True
9,voted_up,True,boolean,boolean,True


root
 |-- author_steamid: long (nullable = true)
 |-- appid: integer (nullable = true)
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- review: string (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- votes_funny: long (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- written_during_early_access: boolean (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)
 |-- language: string (nullable = true)


3. Memory count
Total estimated size: 55.75 GB


**Observation** 
- The full dataset contains **113,885,601 rows**, **18 columns**, and approximately **2.05 billion cells**.
- The estimated in-memory size is approximately **55.74 GB**.
- The schema audit passed: all expected columns are present and their actual Spark data types match the expected schema family.

**Suggested Feature Engineering**
- Keep schema enforcement as an explicit pipeline step.
- Treat `author_steamid` and `appid` as identifiers, not ordinary numeric predictors.
- Treat timestamp fields as Unix-time temporal variables and later convert them into interpretable features such as review date, recency, and time windows.

## Data Preparation - Enforce Schema

In [5]:
df = (
    raw_df.transform(prep.enforce_schema)
        .transform(prep.clean_review_text_basic)
)

## Health Audit - Completeness

In [6]:
## Compeleteness helps us investigate missing value and if we have useful
## features for prediction problems

# 1. Identify missing values per column
# 2. Calculate percentage of missing values
audit.null_report(df, ROW_COUNT).show(truncate=False)

+------------------------------+----------+---------------------+
|column_name                   |null_count|null_rate            |
+------------------------------+----------+---------------------+
|voted_up                      |2956554   |0.02596073580891056  |
|timestamp_created             |2955600   |0.02595235898171183  |
|timestamp_updated             |2414900   |0.021204612161637538 |
|votes_up                      |1772064   |0.015560035548304303 |
|votes_funny                   |1554643   |0.013650917994453048 |
|written_during_early_access   |1423070   |0.012495609519591507 |
|weighted_vote_score           |1378404   |0.012103408928754743 |
|comment_count                 |1233750   |0.010833239576968119 |
|author_steamid                |1878      |1.649023215849737E-5 |
|author_num_games_owned        |1860      |1.6332178815125188E-5|
|author_num_reviews            |1752      |1.5383858754892112E-5|
|appid                         |1291      |1.1335937016304634E-5|
|author_pl

**Observation** 
- Missingness is not negligible for target-like and time-related fields such as `voted_up`, `timestamp_created`, and `timestamp_updated`.
- Since churn analysis depends heavily on time ordering and temporal feature construction, rows missing `timestamp_created` are especially problematic.
- Missing values in voting-related fields may reflect parsing issues, unavailable metadata, or malformed records rather than random absence.

**Suggestions for Data Processing/Feature Engineering**
- Missingness should not be assumed to be completely random without further group-level checking by `appid`, `language`, timestamp period, and possibly review source pattern.
- For churn modeling, remove rows missing critical temporal fields rather than imputing timestamps.

## Data Preparation - Handle Missing Values

In [7]:
df = df.transform(prep.remove_missing)

## Health Audit - Consistency & Validity

In [8]:
## Consistency helps us question "Do columns agree with each other logically"?

# 1. timestamp_created should usually be <= timestamp_updated
# 2. playtime_at_review & playtime_last two weeks should be less than lifetime playtime
# 3. all user in the dataset should have at least 1 numbers of review
# 4. author_playtime_forever at later review time should not be lower than earlier review time
format_report(audit.consistency_report(df, ROW_COUNT))


timestamp_created_gt_timestamp_updated
Violation count: 2
Violation percentage: 1.7561482596908805e-08
Showing first 10 rows:
+-----------------+------+-----------------+-----------------+
|author_steamid   |appid |timestamp_created|timestamp_updated|
+-----------------+------+-----------------+-----------------+
|76561198050620715|931690|1559566943       |1559566933       |
|76561198331063747|218620|1559656828       |1559656824       |
+-----------------+------+-----------------+-----------------+


author_playtime_at_review_gt_author_playtime_forever
Violation count: 43008
Violation percentage: 0.00037764212176392693
Showing first 10 rows:
+-----------------+-----+-------------------------+-----------------------+-----------------+
|author_steamid   |appid|author_playtime_at_review|author_playtime_forever|timestamp_created|
+-----------------+-----+-------------------------+-----------------------+-----------------+
|76561198211357228|2820 |5004                     |5003            

**Observation** 
  - `timestamp_created > timestamp_updated`:
  - **67 violating rows**.
  - This means a review appears to have been created after it was updated, which is logically inconsistent.
- `author_playtime_at_review > author_playtime_forever`:
  - **45,741 violating rows**, about **0.0402%** of the full dataset.
  - This is the most notable playtime consistency issue.
- `author_playtime_last_two_weeks > author_playtime_forever`:
  - **6,417 violating rows**, about **0.00563%**.
  - This is logically inconsistent because recent two-week playtime should not exceed lifetime playtime.
- Users with minimum `author_num_reviews = 0`:
  - **57 users**.
  - This is suspicious because appearing in a review dataset generally implies at least one review.
- `author_playtime_forever` decreases over time for the same user/game sequence:
  - **1 violating row**.
  - This is extremely rare and likely caused by metadata refresh, data correction, or source-level inconsistency.

**Suggestions for Data Processing/Feature Engineering**
- Remove or flag rows where `timestamp_created > timestamp_updated`.
- Remove or flag rows where review-time playtime or two-week playtime exceeds lifetime playtime.
- For temporal features, avoid assuming all user/game records form perfectly monotonic histories.

In [9]:
## Validity helps us question "Do columns within reasonable range"?

# 5. Count-like columns should not be negative
# 6. weighted_vote_score should be between 0 and 1
format_report(audit.validity_report(df, ROW_COUNT))


author_num_games_owned_negative
Violation count: 0
Violation percentage: 0.0
Showing first 10 rows:
+--------------+-----+----------------------+-----------------+
|author_steamid|appid|author_num_games_owned|timestamp_created|
+--------------+-----+----------------------+-----------------+
+--------------+-----+----------------------+-----------------+


author_num_reviews_negative
Violation count: 0
Violation percentage: 0.0
Showing first 10 rows:
+--------------+-----+------------------+-----------------+
|author_steamid|appid|author_num_reviews|timestamp_created|
+--------------+-----+------------------+-----------------+
+--------------+-----+------------------+-----------------+


author_playtime_forever_negative
Violation count: 0
Violation percentage: 0.0
Showing first 10 rows:
+--------------+-----+-----------------------+-----------------+
|author_steamid|appid|author_playtime_forever|timestamp_created|
+--------------+-----+-----------------------+-----------------+
+------

**Observation** 
- Negative values in voting/comment counts are clearly invalid but extremely rare.
- The `weighted_vote_score` issue is more important because it affects over half a million rows.
- Invalid `weighted_vote_score` values may indicate parsing shifts, malformed rows, or source/API artifacts.

**Suggestions for Data Processing/Feature Engineering**
- Remove rows with negative count-like fields.
- Consider adding a data-quality flag if preserving suspicious records for diagnostic EDA.

## Data Preparation - Handle Consistency & Validity

In [10]:
df = (
    # Inconsistent data
    df.transform(prep.remove_timestamp_consistency)
        .transform(prep.remove_playtime_consistency)
        .transform(prep.remove_playtime_forever_decreases_over_time)
    # Invalid data
        .transform(prep.remove_negative_count_values)
        .transform(prep.remove_invalid_weighted_vote_score)
        .transform(prep.remove_invalid_timestamps)
)

## Health Audit - Anomaly & Outliers

In [11]:
## Anomaly indicates suspicious value, not automatically wrong
ano_report = audit.anomaly_report(df, ROW_COUNT)

# 1. Check for outlier value using summary statistics
display(ano_report.get('numeric_describe_df').toPandas())

# 2. Create report regarding anomaly report for suspicious features (1) votes_funny artifact
# 3. Check for instances where votes_funny reached max value
# 4. Check for instances where votes_funny near max value
format_report(ano_report)

# 5. Find the largest vote_funny value that follows right after ARTIFACT_THRESHOLD, compute its propoertion comparing to max value
# If it's between 60% - 95%, it will indicate broader range of artifact from API and we will need to dive deeper
print_section("Largest non-artifact votes_funny value below threshold")
ano_value, ano_percentage = ano_report.get('vote_max_non_artifact', (None, None))
if ano_value is not None: print(ano_value)
if ano_percentage is not None: print(ano_percentage)
del ano_report, ano_value, ano_percentage

,summary,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,votes_up,votes_funny,comment_count,weighted_vote_score
0,count,109161557,109161557,109161557,109161557,109161557,109161557,109161557,109161557,109161557
1,mean,121.48540494892355,24.632843419410005,15738.050532194224,82.62352294956732,7240.072559619134,2.034556176218703,69720.11065609846,0.09793043717762288,0.17396165581775874
2,stddev,462.95620640496344,162.20590065946453,46105.25360755381,550.0086718561357,24762.63944644937,34.762144255671686,1.7304272123829883E7,1.6302031676032218,0.2480964716758437
3,min,0,1,1,0,0,0,0,0,0.0
4,max,33351,10446,6007985,56748,4880175,62199,4294967295,4890,0.99814063



playtime_2w_near_limit
Violation count: 16388
Violation percentage: 0.00014389878839907074
Showing first 10 rows:
+-----------------+------+------------------------------+-----------------------+-----------------+
|author_steamid   |appid |author_playtime_last_two_weeks|author_playtime_forever|timestamp_created|
+-----------------+------+------------------------------+-----------------------+-----------------+
|76561199404792985|730   |56748                         |203490                 |1666940962       |
|76561199488329913|730   |45285                         |114381                 |1691684617       |
|76561199013645279|730   |41953                         |81887                  |1603308881       |
|76561199177996816|730   |36370                         |181325                 |1626262465       |
|76561198315604942|431960|33369                         |296485                 |1673236213       |
|76561198844172674|730   |33012                         |625390                 |1580

**Observation** 
- Several numeric columns show extreme maximum values, including:
  - `author_num_games_owned` maximum: **1,651,393,883**.
  - `author_num_reviews` maximum: **1,651,393,883**.
  - `author_playtime_forever` maximum: **1,403,342,723**.
  - `author_playtime_last_two_weeks` maximum: **1,393,862,250**.
  - `author_playtime_at_review` maximum: **1,402,630,204**.
  - `votes_up` maximum: **1,699,017,189**.
  - `comment_count` maximum: **1,699,015,475**.
- These extreme values are not plausible as normal behavioral values and likely reflect malformed records, parsing errors, or source-level artifacts.
- `author_playtime_last_two_weeks` has a physical upper-bound issue:
  - **16,632 rows** are near the two-week playtime limit.
  - **17 rows** exceed the two-week limit.
- `votes_funny` has a clear unsigned integer / API artifact pattern:
  - **1,797 rows** are detected as `votes_funny` uint32-style artifacts.
  - **1,511 rows** are exactly `4,294,967,295`, which equals `2^32 - 1`.
  - **286 rows** are near the exact uint32 maximum.
  - The largest non-artifact `votes_funny` value below the threshold is **1,699,020,607**, which is still suspiciously large.
- The full audit reveals much stronger anomaly patterns than the sample audit because rare artifacts are easier to detect with 113M rows.
- The `votes_funny` artifact is the clearest outlier problem because values around `2^32 - 1` are not realistic engagement counts.
- Some two-week playtime values near the maximum may be technically possible if Steam records app runtime rather than active human play. However, values exceeding the physical two-week limit should be removed.
- Very large maxima in multiple columns suggest that some rows may be shifted/malformed rather than merely heavy-tailed.


**Suggestions for Data Processing/Feature Engineering**
- Remove impossible `author_playtime_last_two_weeks` values that exceed the two-week limit.
- Remove `votes_funny` values matching or near the uint32 artifact range.
- Use transformations for heavy-tailed but valid count fields, such as `log1p(votes_up)`, `log1p(comment_count)`, and `log1p(playtime)`.
- For final modeling, consider winsorization or robust binning for valid but extreme engagement/playtime values.

## Data Preparation - Handle Abnormal Data

In [12]:
df = (
    df.transform(prep.remove_impossible_two_week_playtime)
        .transform(prep.remove_impossible_vote_funny)
)
ROW_COUNT = df.count()

## Health Audit - Noise

In [13]:
## Noises are values that are valid but possibly not useful & unstable

# 1. Check for columns with mostly zeros
audit.noise_report(df, ROW_COUNT).show(truncate=False)

+------------------------------+-------------------+---------------------+
|column_name                   |zero_or_false_count|zero_or_false_rate   |
+------------------------------+-------------------+---------------------+
|comment_count                 |104939754          |0.9613409049626817   |
|author_playtime_last_two_weeks|96420326           |0.883295413991887    |
|votes_funny                   |96334133           |0.8825058099241908   |
|votes_up                      |76730287           |0.7029172523372438   |
|weighted_vote_score           |72520149           |0.6643486668330527   |
|author_num_games_owned        |54365624           |0.49803717068958336  |
|author_playtime_at_review     |131541             |0.0012050318316897913|
|author_num_reviews            |0                  |0.0                  |
|author_playtime_forever       |0                  |0.0                  |
+------------------------------+-------------------+---------------------+



**Observations**
- `comment_count`:
  - **107,639,931 zero rows**, about **94.52%**.
  - Most reviews receive no comments.
- `author_playtime_last_two_weeks`:
  - **100,949,251 zero rows**, about **88.64%**.
  - This is important for churn because recent inactivity is directly related to churn-like behavior.
- `votes_funny`:
  - **97,979,357 zero rows**, about **86.03%**.
  - Funny votes are rare and should likely be treated as sparse engagement signal.
- `votes_up`:
  - **77,654,621 zero rows**, about **68.19%**.
  - Raw vote counts may be less useful than transformed or indicator-based versions.
- `weighted_vote_score`:
  - **73,910,148 zero rows**, about **64.90%**.
  - A zero value may mean no meaningful voting activity, unavailable score, or low engagement.
- `author_num_games_owned`:
  - **56,725,784 zero rows**, about **49.81%**.
  - This is suspicious from a behavioral perspective and should be interpreted as “zero recorded owned games,” not necessarily true zero ownership.
- `author_playtime_at_review` and `author_playtime_forever` have relatively low zero rates:
  - `author_playtime_at_review`: about **1.64%** zero.
  - `author_playtime_forever`: about **1.53%** zero.

**Suggestions for Data Processing/Feature Engineering**
- Create binary indicators for sparse engagement features:
  - `has_comments = comment_count > 0`
  - `has_votes_up = votes_up > 0`
  - `has_funny_votes = votes_funny > 0`
  - `has_recent_playtime = author_playtime_last_two_weeks > 0`
- Use log transformations for valid non-negative count variables.

## Health Audit - Uniqueness & Duplicates

In [ ]:
## Uniqueness of catgorical features tells us insight of categories, while for numerical features,
## it indicates potential few_unique_values_per_column issue or possibility for discretization to a categorical type

# 1. Numbers of unique values for categorical/numerical & ID columns
audit.uniqueness_report(df, ROW_COUNT).show(truncate=False)

## Duplicate tells us how many rows has exact duplicates and how frequent would a user to review the same game

# 2. Numbers of duplicate found in dataframe
# 3. Does the same user appear to have created multiple review records for the same game at the same time?
# 4. Inspect more details ("author_steamid", "appid", "timestamp_created") regarding instances in #3
# 5. Print out the duplicate reviews
dup_reports = audit.duplicate_report(df, ROW_COUNT)
for (report, issue_df) in zip(dup_reports['summary_rows'], dup_reports['issue_dfs'].values()):
    print_section(report['issue_name'])
    for name, val in report.items():
        if name == 'issue_name': continue
        print(f"{name}: {val}")
    issue_df.show(n=5, truncate=False)

# 6. Create report on metadata difference on same (author ID, game ID, review date created)
# UN-ACHIEVE-ABLE; reason: too much join, putting too pressure on spark worker memory

+------------------------------+--------+---------------------+
|column_name                   |n_unique|unique_rate          |
+------------------------------+--------+---------------------+
|voted_up                      |2       |1.832176783952975E-8 |
|written_during_early_access   |2       |1.832176783952975E-8 |
|comment_count                 |457     |4.1865239513325475E-6|
|author_num_reviews            |1290    |1.1817540256496689E-5|
|votes_funny                   |3022    |2.768419120552945E-5 |
|votes_up                      |4343    |3.978571886353885E-5 |
|author_num_games_owned        |9676    |8.864071280764493E-5 |
|author_playtime_last_two_weeks|19894   |1.8224662469980242E-4|
|author_playtime_at_review     |356750  |0.003268145338376119 |
|author_playtime_forever       |540220  |0.0049488927111353805|
|weighted_vote_score           |5415579 |0.04961149057731634  |
+------------------------------+--------+---------------------+



**Observations**
- Exact duplicates are rare relative to the full dataset but should still be removed because they provide no additional information.
- Composite duplicate review keys are more analytically important because they may represent the same review event appearing multiple times.
- Some composite duplicates may reflect updated metadata, changed vote counts, changed text, language-label noise, or repeated extraction of the same review event.
- Null-key duplicate groups should largely disappear after missing-value removal.

**Suggestions for Data Processing/Feature Engineering**
- Remove exact duplicate rows.
- Resolve duplicate review keys deterministically, preferably by keeping the most recently updated version using `timestamp_updated`.

## Data Preparation - Handle Duplicates

In [ ]:
df = (
    df.transform(prep.remove_duplicate_rows)
        .transform(prep.remove_duplicate_reviews)
)

## Write Cleaned Full Data to Parquet File

In [ ]:
print("Final cleaned schema before writing:")
df.printSchema()

write_spark_parquet(df=df, path=paths.cleaned_parquet_spark)

In [ ]:
spark.stop()